In [ ]:
# ================================================================
# GENERAL 1D STEADY HEAT CONDUCTION USING FDM
# ================================================================
#
# Governing equation:
#
#              k d²T/dx² + S = 0
#
#
# LEFT BOUNDARY:
#
#              T(0) = T_left
#
#
# RIGHT BOUNDARY:
#
# The user may choose either:
#
#   1. Prescribed temperature (Dirichlet BC)
#
#              T(L) = T_right
#
#   OR
#
#   2. Insulated boundary (Neumann BC)
#
#              dT/dx(L) = 0
#
#
# Assumptions:
#
#   1. Steady-state conduction
#   2. One-dimensional heat transfer
#   3. Constant thermal conductivity k
#   4. Uniform volumetric heat generation S
#
# ================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ================================================================
# 1. USER INPUTS
# ================================================================
#
# CHANGE THESE VALUES FOR DIFFERENT 1D PROBLEMS
# ================================================================

L = 1.0             # Length of wall/domain (m)

N = 10               # Total number of physical grid nodes

T_left = 100.0      # Temperature at x = 0 (°C)

k = 10.0             # Thermal conductivity (W/m.K)

S = 100.0             # Volumetric heat generation (W/m^3)


# ================================================================
# 2. SELECT RIGHT BOUNDARY CONDITION
# ================================================================
#
# Choose ONE of the following:
#
#       right_BC = "temperature"
#
# OR
#
#       right_BC = "insulated"
#
#
# IMPORTANT:
#
# If right_BC = "temperature":
#       T_right is prescribed by the user.
#
# If right_BC = "insulated":
#       T_right is NOT prescribed.
#       The program calculates T(L).
#
# ================================================================




#right_BC = "temperature"
right_BC = "insulated"



# This value is used ONLY when:
#
#       right_BC = "temperature"
#
# It is automatically ignored when:
#
#       right_BC = "insulated"

#T_right = 200.0


# ================================================================
# 3. BASIC INPUT CHECKS
# ================================================================

if N < 3:
    raise ValueError("N must be at least 3.")

if L <= 0:
    raise ValueError("L must be positive.")

if k <= 0:
    raise ValueError("Thermal conductivity k must be positive.")

if right_BC not in ["temperature", "insulated"]:
    raise ValueError(
        'right_BC must be either "temperature" or "insulated".'
    )


# ================================================================
# 4. COMPUTATIONAL GRID
# ================================================================

dx = L / (N - 1)

x = np.linspace(0, L, N)


print("================================================")
print("GRID INFORMATION")
print("================================================")

print("Domain length L         =", L, "m")
print("Total number of nodes N =", N)
print("Grid spacing dx         =", dx, "m")
print("Node positions          =", x)

print()


# ================================================================
# 5. DISPLAY PHYSICAL PROBLEM
# ================================================================

print("================================================")
print("PHYSICAL PROBLEM")
print("================================================")

print("Thermal conductivity k =", k, "W/(m.K)")
print("Heat generation S      =", S, "W/m³")
print("Left temperature       =", T_left, "°C")

print()


if right_BC == "temperature":

    print("Right boundary type    = Prescribed temperature")
    print("Right temperature      =", T_right, "°C")


elif right_BC == "insulated":

    print("Right boundary type    = Insulated")
    print("Boundary condition     = dT/dx(L) = 0")
    print("Right temperature      = UNKNOWN (calculated by FDM)")


print()


# ================================================================
# 6. FINITE DIFFERENCE EQUATION
# ================================================================
#
# Governing equation:
#
#              k d²T/dx² + S = 0
#
#
# Divide by k:
#
#              d²T/dx² = -S/k
#
#
# Central difference:
#
#       T(i-1) - 2T(i) + T(i+1)
#       -------------------------
#                  dx²
#
#              = -S/k
#
#
# Therefore:
#
#       T(i-1) - 2T(i) + T(i+1)
#
#              = -S dx²/k
#
#
# Multiplying by -1:
#
#       2T(i) - T(i-1) - T(i+1)
#
#              = S dx²/k
#
# ================================================================


source_term = S * dx**2 / k


print("================================================")
print("FINITE DIFFERENCE EQUATION")
print("================================================")

print(
    "2T(i) - T(i-1) - T(i+1) = S dx²/k"
)

print()

print(
    "Source contribution S dx²/k =",
    source_term
)

print()


# ================================================================
# 7. BUILD MATRIX SYSTEM
#
#                         A T = b
# ================================================================


# ================================================================
# CASE 1:
# PRESCRIBED RIGHT TEMPERATURE
# ================================================================
#
# Boundary conditions:
#
#       T(0) = T_left
#
#       T(L) = T_right
#
#
# Both boundary temperatures are known.
#
# Unknowns:
#
#       T1, T2, ..., T(N-2)
#
# Therefore:
#
#       Number of unknowns = N - 2
#
# ================================================================

if right_BC == "temperature":


    num_unknowns = N - 2


    A = np.zeros(
        (num_unknowns, num_unknowns)
    )


    b = np.full(
        num_unknowns,
        source_term
    )


    # ------------------------------------------------------------
    # Construct FDM coefficient matrix
    # ------------------------------------------------------------

    for i in range(num_unknowns):


        # Central coefficient

        A[i, i] = 2.0


        # Left neighboring unknown

        if i > 0:

            A[i, i - 1] = -1.0


        # Right neighboring unknown

        if i < num_unknowns - 1:

            A[i, i + 1] = -1.0


    # ------------------------------------------------------------
    # Apply known boundary temperatures
    # ------------------------------------------------------------

    b[0] += T_left

    b[-1] += T_right



# ================================================================
# CASE 2:
# INSULATED RIGHT BOUNDARY
# ================================================================
#
# Boundary conditions:
#
#       T(0) = T_left
#
#       dT/dx(L) = 0
#
#
# IMPORTANT:
#
# The right-boundary temperature is UNKNOWN.
#
# Therefore:
#
#       T1, T2, ..., T(N-2), T(N-1)
#
# are all unknown.
#
# Number of unknowns:
#
#       N - 1
#
#
# ------------------------------------------------
# GHOST NODE TREATMENT
# ------------------------------------------------
#
# Introduce a ghost node outside the right boundary.
#
# At x = L:
#
#       dT/dx = 0
#
#
# Central difference:
#
#       T_ghost - T(N-2)
#       ---------------- = 0
#              2dx
#
#
# Therefore:
#
#       T_ghost = T(N-2)
#
#
# Applying the governing FDM equation at
# the right-boundary node gives:
#
#       -2T(N-2) + 2T(N-1)
#
#                 = S dx²/k
#
# ================================================================

elif right_BC == "insulated":


    num_unknowns = N - 1


    A = np.zeros(
        (num_unknowns, num_unknowns)
    )


    b = np.full(
        num_unknowns,
        source_term
    )


    # ------------------------------------------------------------
    # Ordinary nodes:
    #
    # T1 through T(N-2)
    # ------------------------------------------------------------

    for i in range(num_unknowns - 1):


        # Central coefficient

        A[i, i] = 2.0


        # Left neighbor

        if i > 0:

            A[i, i - 1] = -1.0


        # Right neighbor

        A[i, i + 1] = -1.0


    # ------------------------------------------------------------
    # Left prescribed-temperature boundary
    # ------------------------------------------------------------

    b[0] += T_left


    # ------------------------------------------------------------
    # Last row:
    # insulated right boundary
    #
    #       -2T(N-2) + 2T(N-1)
    #
    #              = S dx²/k
    # ------------------------------------------------------------

    A[-1, -2] = -2.0

    A[-1, -1] = 2.0


# ================================================================
# 8. DISPLAY NUMBER OF UNKNOWNS
# ================================================================

print("================================================")
print("NUMBER OF UNKNOWN TEMPERATURES")
print("================================================")

print(num_unknowns)

print()


# ================================================================
# 9. DISPLAY MATRIX SYSTEM
# ================================================================

if N <= 12:

    print("================================================")
    print("COEFFICIENT MATRIX A")
    print("================================================")

    print(A)

    print()


    print("================================================")
    print("RIGHT-HAND SIDE VECTOR b")
    print("================================================")

    print(b)

    print()


# ================================================================
# 10. SOLVE LINEAR SYSTEM
# ================================================================
#
# FDM has already converted the differential equation into:
#
#                         A T = b
#
#
# np.linalg.solve is NOT the Finite Difference Method.
#
# FDM created A and b.
#
# np.linalg.solve only solves the resulting
# algebraic system.
#
# ================================================================

T_unknown = np.linalg.solve(
    A,
    b
)


print("================================================")
print("CALCULATED UNKNOWN TEMPERATURES")
print("================================================")

print(T_unknown)

print()


# ================================================================
# 11. BUILD COMPLETE TEMPERATURE VECTOR
# ================================================================

T_fdm = np.zeros(N)


# Left boundary is prescribed in both cases

T_fdm[0] = T_left


# ------------------------------------------------
# CASE 1:
# Prescribed right temperature
# ------------------------------------------------

if right_BC == "temperature":

    T_fdm[1:-1] = T_unknown

    T_fdm[-1] = T_right


# ------------------------------------------------
# CASE 2:
# Insulated right boundary
# ------------------------------------------------

elif right_BC == "insulated":

    # T1 through T(N-1) were calculated

    T_fdm[1:] = T_unknown


    # The final value is the calculated
    # right-boundary temperature

    T_right_calculated = T_fdm[-1]


# ================================================================
# 12. ANALYTICAL SOLUTION
# ================================================================


# ================================================================
# CASE 1:
# PRESCRIBED TEMPERATURE AT BOTH ENDS
# ================================================================
#
# Boundary conditions:
#
#       T(0) = T_left
#
#       T(L) = T_right
#
#
# Analytical solution:
#
#       T(x)
#
#       = T_left
#
#       + (T_right - T_left)x/L
#
#       + (S/(2k))x(L-x)
#
# ================================================================

if right_BC == "temperature":

    T_exact = (

        T_left

        + (T_right - T_left)
        * x / L

        + (S / (2 * k))
        * x
        * (L - x)

    )


# ================================================================
# CASE 2:
# INSULATED RIGHT BOUNDARY
# ================================================================
#
# Boundary conditions:
#
#       T(0) = T_left
#
#       dT/dx(L) = 0
#
#
# Analytical solution:
#
#       T(x)
#
#       = T_left
#
#       + (S/k)(Lx - x²/2)
#
# ================================================================

elif right_BC == "insulated":

    T_exact = (

        T_left

        + (S / k)

        * (
            L * x
            - x**2 / 2
        )

    )


# ================================================================
# 13. ERROR
# ================================================================

abs_error = np.abs(
    T_fdm - T_exact
)


# ================================================================
# 14. RESULTS TABLE
# ================================================================

results = pd.DataFrame({

    "Node i":
        np.arange(N),

    "x (m)":
        np.round(x, 5),

    "FDM T (°C)":
        np.round(T_fdm, 5),

    "Exact T (°C)":
        np.round(T_exact, 5),

    "Absolute Error (°C)":
        np.round(abs_error, 8)

})


print("================================================")
print("FDM VS ANALYTICAL RESULTS")
print("================================================")

display(results)


# ================================================================
# 15. COMPLETE TEMPERATURE VECTOR
# ================================================================

print("================================================")
print("COMPLETE FDM TEMPERATURE VECTOR")
print("================================================")

print(T_fdm)

print()


# ================================================================
# 16. RIGHT-BOUNDARY INFORMATION
# ================================================================

print("================================================")
print("RIGHT-BOUNDARY RESULT")
print("================================================")


if right_BC == "temperature":

    print(
        "Boundary type:"
    )

    print(
        "Prescribed temperature (Dirichlet)"
    )

    print()

    print(
        f"T(L) was prescribed as {T_right:.6f} °C"
    )


elif right_BC == "insulated":

    print(
        "Boundary type:"
    )

    print(
        "Insulated boundary (Neumann)"
    )

    print()

    print(
        "dT/dx(L) = 0"
    )

    print()

    print(
        "T(L) was NOT prescribed."
    )

    print(
        f"FDM calculated T(L) = "
        f"{T_right_calculated:.6f} °C"
    )


print()


# ================================================================
# 17. ANALYTICAL CURVE FOR PLOTTING
# ================================================================

x_exact = np.linspace(
    0,
    L,
    300
)


# ------------------------------------------------
# Prescribed right temperature
# ------------------------------------------------

if right_BC == "temperature":

    T_exact_plot = (

        T_left

        + (T_right - T_left)
        * x_exact / L

        + (S / (2 * k))
        * x_exact
        * (L - x_exact)

    )


# ------------------------------------------------
# Insulated right boundary
# ------------------------------------------------

elif right_BC == "insulated":

    T_exact_plot = (

        T_left

        + (S / k)

        * (
            L * x_exact
            - x_exact**2 / 2
        )

    )


# ================================================================
# 18. PLOT
# ================================================================
#
# Analytical solution:
#
#       Continuous line
#
#
# FDM solution:
#
#       Discrete nodal points only
#
# ================================================================

plt.figure(
    figsize=(9, 5)
)


# ------------------------------------------------
# Analytical solution
# ------------------------------------------------

plt.plot(
    x_exact,
    T_exact_plot,
    linewidth=2,
    label="Analytical Solution"
)


# ------------------------------------------------
# FDM discrete nodal solution
# ------------------------------------------------

plt.scatter(
    x,
    T_fdm,
    s=80,
    c="darkred",
    label=f"FDM Solution (N = {N})"
)


# ------------------------------------------------
# Axis labels
# ------------------------------------------------

plt.xlabel(
    "Position x (m)"
)

plt.ylabel(
    "Temperature T (°C)"
)


# ------------------------------------------------
# Boundary-condition information for title
# ------------------------------------------------

if right_BC == "temperature":

    BC_text = (
        f"T(0) = {T_left} °C, "
        f"T(L) = {T_right} °C"
    )


elif right_BC == "insulated":

    BC_text = (
        f"T(0) = {T_left} °C, "
        f"dT/dx(L) = 0 (Insulated)"
    )


plt.title(

    "1D Steady Heat Conduction\n"

    + BC_text

    + f"\nL = {L} m, N = {N}, "
      f"dx = {dx:.4f} m, "
      f"k = {k}, S = {S}"
)


plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()